# 00. Tổng quan Dữ liệu (Data Overview)

Mục tiêu: Đánh giá tổng quan toàn bộ 8 bảng dữ liệu Dunnhumby theo chuẩn hợp đồng:
- **Structure**: Kích thước bảng (rows, cols), memory usage, danh sách cột và kiểu dữ liệu
- **Grain**: Kiểm định candidate grain, số nhóm trùng lặp (duplicate groups), excess rows, tính duy nhất (`is_unique`), và tính hợp lệ (`is_valid`)
- **Key**: Phân tích các khóa định danh (cardinality, missingness, uniqueness, validity)
- **Time Coverage**: Đánh giá độ phủ thời gian có ngữ nghĩa (ordinal DAY/WEEK_NO, clock code TRANS_TIME, interval START_DAY/END_DAY)
- **Quality cơ bản**: Tỷ lệ missing, blank string, cột hằng số (constant), và trùng lặp toàn dòng (exact duplicate rows)
- **Numeric Distribution**: Thống kê phân phối các cột định lượng (count, mean, std, min, p25, median, p75, max) tương đương `df.describe()`

Toàn bộ logic được đóng gói trong package `propensity.overview`. Notebook chỉ gọi orchestrator và hiển thị kết quả.

In [2]:
from pathlib import Path
from propensity.overview import build_dunnhumby_overview

In [3]:
raw_dir = Path("../data/raw/Dunnhumby")
report = build_dunnhumby_overview(raw_dir=raw_dir)

## 1. Table Summary (Structure)

In [4]:
report.table_summary

,table,n_rows,n_columns,memory_bytes
0,transaction,2595732,12,259573332
1,product,92353,7,10190962
2,demographic,801,8,87189
3,campaign_desc,30,4,1272
4,campaign_table,7208,3,223580
5,coupon,124548,3,3362928
6,coupon_redempt,2318,4,81262
7,causal,36786524,5,1618607188


## 2. Column Summary

In [5]:
report.column_summary

,table,column,dtype,role,is_required,is_identifier,is_grain,is_time
0,transaction,household_key,Int64,entity,True,True,False,False
1,transaction,BASKET_ID,Int64,id,True,True,True,False
2,transaction,DAY,int64,time,True,False,False,True
3,transaction,PRODUCT_ID,Int64,id,True,True,True,False
4,transaction,QUANTITY,int64,numeric,True,False,False,False
5,transaction,SALES_VALUE,float64,numeric,True,False,False,False
6,transaction,STORE_ID,Int64,id,True,True,False,False
7,transaction,RETAIL_DISC,float64,numeric,False,False,False,False
8,transaction,TRANS_TIME,int64,time,True,False,False,True
9,transaction,WEEK_NO,int64,time,True,False,False,True


## 3. Candidate Grain Summary

In [6]:
report.grain_summary

,table,grain,n_rows,complete_key_rows,missing_key_rows,unique_groups,duplicate_groups,excess_rows,max_rows_per_group,is_unique,is_valid
0,transaction,"BASKET_ID, PRODUCT_ID",2595732,2595732,0,2595732,0,0,1,True,True
1,product,PRODUCT_ID,92353,92353,0,92353,0,0,1,True,True
2,demographic,household_key,801,801,0,801,0,0,1,True,True
3,campaign_desc,CAMPAIGN,30,30,0,30,0,0,1,True,True
4,campaign_table,"CAMPAIGN, household_key",7208,7208,0,7208,0,0,1,True,True
5,coupon,"COUPON_UPC, PRODUCT_ID, CAMPAIGN",124548,124548,0,117560,1824,5164,19,False,False
6,coupon_redempt,"household_key, DAY, COUPON_UPC, CAMPAIGN",2318,2318,0,2318,0,0,1,True,True
7,causal,"PRODUCT_ID, STORE_ID, WEEK_NO",36786524,36786524,0,36756034,15245,15245,2,False,False


## 4. Identifier Key Summary

In [7]:
report.identifier_summary

,table,column,n_rows,n_missing,missing_pct,n_unique_non_missing,is_unique,is_valid
0,transaction,household_key,2595732,0,0.0,2500,False,False
1,transaction,BASKET_ID,2595732,0,0.0,276484,False,False
2,transaction,PRODUCT_ID,2595732,0,0.0,92339,False,False
3,transaction,STORE_ID,2595732,0,0.0,582,False,False
4,product,PRODUCT_ID,92353,0,0.0,92353,True,True
5,product,MANUFACTURER,92353,0,0.0,6476,False,False
6,demographic,household_key,801,0,0.0,801,True,True
7,campaign_desc,CAMPAIGN,30,0,0.0,30,True,True
8,campaign_table,CAMPAIGN,7208,0,0.0,30,False,False
9,campaign_table,household_key,7208,0,0.0,1584,False,False


## 5. Time Coverage Summary

In [8]:
report.time_summary

,table,column,kind,unit,start_column,end_column,min,max,n_unique_non_missing,span,n_missing,missing_endpoint_rows,start_after_end_rows,invalid_clock_rows
0,transaction,DAY,ordinal,day_index,<NA>,<NA>,1,711,711,710,0,<NA>,<NA>,<NA>
1,transaction,WEEK_NO,ordinal,week,<NA>,<NA>,1,102,102,101,0,<NA>,<NA>,<NA>
2,transaction,TRANS_TIME,clock,HHMM,<NA>,<NA>,0,2359,1440,<NA>,0,<NA>,<NA>,0
3,campaign_desc,START_DAY,ordinal,day,<NA>,<NA>,224,659,27,435,0,<NA>,<NA>,<NA>
4,campaign_desc,END_DAY,ordinal,day,<NA>,<NA>,264,719,28,455,0,<NA>,<NA>,<NA>
5,campaign_desc,<NA>,interval,day,START_DAY,END_DAY,224,719,<NA>,495,0,0,0,<NA>
6,coupon_redempt,DAY,ordinal,day_index,<NA>,<NA>,225,704,328,479,0,<NA>,<NA>,<NA>
7,causal,WEEK_NO,ordinal,week,<NA>,<NA>,9,101,93,92,0,<NA>,<NA>,<NA>


## 6. Basic Quality Summary

In [9]:
report.quality_summary

,table,column,n_rows,n_missing,n_blank,missing_pct,n_unique_non_missing,is_all_missing,is_constant,duplicate_groups,excess_rows,duplicate_pct
0,transaction,household_key,2595732,0,0,0.0,2500,False,False,<NA>,<NA>,<NA>
1,transaction,BASKET_ID,2595732,0,0,0.0,276484,False,False,<NA>,<NA>,<NA>
2,transaction,DAY,2595732,0,0,0.0,711,False,False,<NA>,<NA>,<NA>
3,transaction,PRODUCT_ID,2595732,0,0,0.0,92339,False,False,<NA>,<NA>,<NA>
4,transaction,QUANTITY,2595732,0,0,0.0,11746,False,False,<NA>,<NA>,<NA>
5,transaction,SALES_VALUE,2595732,0,0,0.0,5502,False,False,<NA>,<NA>,<NA>
6,transaction,STORE_ID,2595732,0,0,0.0,582,False,False,<NA>,<NA>,<NA>
7,transaction,RETAIL_DISC,2595732,0,0,0.0,2414,False,False,<NA>,<NA>,<NA>
8,transaction,TRANS_TIME,2595732,0,0,0.0,1440,False,False,<NA>,<NA>,<NA>
9,transaction,WEEK_NO,2595732,0,0,0.0,102,False,False,<NA>,<NA>,<NA>


## 7. Numeric Distribution Summary (df.describe)

Thống kê phân phối các cột số (count, mean, std, min, p25, median, p75, max) tương đương `df.describe()`.

In [10]:
report.numeric_summary

,table,column,count,mean,std,min,p25,median,p75,max
0,transaction,QUANTITY,2595732,100.428558,1153.436211,0.0,1.0,1.0,1.0,89638.0
1,transaction,SALES_VALUE,2595732,3.10412,4.182274,0.0,1.29,2.0,3.49,840.0
2,transaction,RETAIL_DISC,2595732,-0.538705,1.249191,-180.0,-0.69,-0.01,0.0,3.99
3,transaction,COUPON_DISC,2595732,-0.016416,0.216841,-55.93,0.0,0.0,0.0,0.0
4,transaction,COUPON_MATCH_DISC,2595732,-0.002919,0.03969,-7.7,0.0,0.0,0.0,0.0


## 8. Provenance

In [11]:
report.provenance

,table,path,size_bytes,scan_mode,chunk_rows,exact,rows_scanned,spill_partitions,max_chunk_rows,max_partition_rows
0,transaction,..\data\raw\Dunnhumby\transaction_data.csv,141742346,file,250000,True,2595732,64,2595732,<NA>
1,product,..\data\raw\Dunnhumby\product.csv,6429896,file,250000,True,92353,64,92353,<NA>
2,demographic,..\data\raw\Dunnhumby\hh_demographic.csv,44349,file,250000,True,801,64,801,<NA>
3,campaign_desc,..\data\raw\Dunnhumby\campaign_desc.csv,540,file,250000,True,30,64,30,<NA>
4,campaign_table,..\data\raw\Dunnhumby\campaign_table.csv,95874,file,250000,True,7208,64,7208,<NA>
5,coupon,..\data\raw\Dunnhumby\coupon.csv,2822804,file,250000,True,124548,64,124548,<NA>
6,coupon_redempt,..\data\raw\Dunnhumby\coupon_redempt.csv,54108,file,250000,True,2318,64,2318,<NA>
7,causal,..\data\raw\Dunnhumby\causal_data.csv,695858427,file,250000,True,36786524,64,36786524,<NA>
